In [ ]:
from datasets import load_dataset

ds = load_dataset("danidanou/Bloomberg_Financial_News")

big_five_banks = [
    "Scotiabank",
    "Royal Bank",
    "Toronto-Dominion",
    "Bank of Montreal",
    "Canadian Imperial Bank of Commerce",
]
counter = 0
dates = []
for split in ds:
    for example in ds[split]:
        text = example.get("text", "") or example.get("headline", "") or str(example)
        if any(bank.lower() in text.lower() for bank in big_five_banks):
            # print(f"[{split}] {text}")
            counter += 1
            dates.append(example.get("Date", "").strftime("%Y-%m-%d"))

In [6]:
# get news article from `ds` for this particular date
target_date = "2023-01-01"  # example date
articles_on_target_date = []
for split in ds:
    for example in ds[split]:
        date_str = example.get("Date", "").strftime("%Y-%m-%d")
        if date_str == target_date:
            articles_on_target_date.append(example)
print(f"Number of articles mentioning big five banks: {counter}")


Number of articles mentioning big five banks: 10456


In [8]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("danidanou/Bloomberg_Financial_News")
df = ds["train"].to_pandas()

banks = {
    "RBC": ["Royal Bank of Canada", "RBC"],
    "TD": ["Toronto-Dominion", "TD Bank"],
    "Scotiabank": ["Scotiabank", "Bank of Nova Scotia"],
    "BMO": ["Bank of Montreal", "BMO"],
    "CIBC": ["CIBC", "Canadian Imperial"],
}

df["date_only"] = pd.to_datetime(df["Date"]).dt.date
for bank_name, keywords in banks.items():
    df[bank_name] = df.apply(
        lambda row: any(
            kw.lower() in str(row.get("Headline", "")).lower() or kw.lower() in str(row.get("Article", "")).lower()
            for kw in keywords
        ),
        axis=1,
    )
bank_cols = list(banks.keys())

# Find dates with all 5, ranked by minimum bank count (best balance)
date_stats = df.groupby("date_only")[bank_cols].sum()
date_has_all = date_stats[(date_stats > 0).all(axis=1)]
date_has_all["min_count"] = date_has_all.min(axis=1)
date_has_all["total"] = date_has_all[bank_cols].sum(axis=1)

# Top 20 by most balanced coverage (highest minimum)
best = date_has_all.sort_values(["min_count", "total"], ascending=False).head(20)
print("TOP 20 dates by balanced coverage:")
print()
for d, row in best.iterrows():
    print(
        f"{d}: RBC={int(row['RBC']):2d}  TD={int(row['TD']):2d}  "
        f"Scotia={int(row['Scotiabank']):2d}  BMO={int(row['BMO']):2d}  "
        f"CIBC={int(row['CIBC']):2d}  | total={int(row['total'])}  min={int(row['min_count'])}"
    )

TOP 20 dates by balanced coverage:

2010-12-02: RBC=11  TD= 8  Scotia= 4  BMO= 7  CIBC= 7  | total=37  min=4
2010-12-07: RBC=12  TD= 7  Scotia= 5  BMO= 7  CIBC= 4  | total=35  min=4
2013-05-29: RBC=11  TD= 4  Scotia= 5  BMO= 9  CIBC= 5  | total=34  min=4
2011-09-01: RBC=12  TD= 6  Scotia= 6  BMO= 5  CIBC= 4  | total=33  min=4
2012-08-30: RBC=11  TD= 5  Scotia= 7  BMO= 6  CIBC= 4  | total=33  min=4
2012-08-01: RBC= 9  TD= 4  Scotia= 4  BMO= 5  CIBC= 4  | total=26  min=4
2012-12-06: RBC=10  TD= 4  Scotia= 4  BMO= 4  CIBC= 4  | total=26  min=4
2010-12-03: RBC=12  TD= 6  Scotia= 5  BMO= 4  CIBC= 3  | total=30  min=3
2011-12-01: RBC= 5  TD= 8  Scotia= 3  BMO= 7  CIBC= 6  | total=29  min=3
2013-05-30: RBC= 6  TD= 5  Scotia= 3  BMO= 5  CIBC= 8  | total=27  min=3
2013-05-23: RBC= 9  TD= 3  Scotia= 3  BMO= 7  CIBC= 3  | total=25  min=3
2010-06-15: RBC= 6  TD= 4  Scotia= 4  BMO= 5  CIBC= 3  | total=22  min=3
2012-11-29: RBC= 9  TD= 3  Scotia= 3  BMO= 3  CIBC= 4  | total=22  min=3
2012-07-27: RBC

/tmp/ipykernel_2102/3963835949.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  date_has_all["min_count"] = date_has_all.min(axis=1)
/tmp/ipykernel_2102/3963835949.py:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  date_has_all["total"] = date_has_all[bank_cols].sum(axis=1)


In [ ]:
from collections import Counter

count_data = Counter(dates)
count_data.most_common()

[('2013-02-28', 31),
 ('2010-08-06', 30),
 ('2010-09-30', 28),
 ('2010-10-14', 28),
 ('2010-11-04', 28),
 ('2010-10-13', 27),
 ('2010-07-19', 27),
 ('2013-11-01', 26),
 ('2010-11-16', 26),
 ('2010-10-18', 26),
 ('2010-11-12', 24),
 ('2010-07-12', 24),
 ('2010-11-30', 24),
 ('2010-11-02', 24),
 ('2012-01-12', 23),
 ('2010-10-22', 23),
 ('2010-10-12', 23),
 ('2011-12-01', 23),
 ('2012-06-29', 23),
 ('2010-07-15', 23),
 ('2013-03-21', 22),
 ('2012-10-15', 22),
 ('2010-10-26', 22),
 ('2012-03-27', 22),
 ('2010-07-20', 22),
 ('2010-12-17', 22),
 ('2010-11-10', 22),
 ('2010-09-24', 22),
 ('2010-10-05', 22),
 ('2011-10-19', 22),
 ('2011-10-07', 21),
 ('2011-07-20', 21),
 ('2011-04-07', 21),
 ('2013-02-26', 21),
 ('2013-02-27', 21),
 ('2010-10-25', 21),
 ('2010-10-07', 21),
 ('2010-08-02', 21),
 ('2010-07-21', 21),
 ('2010-07-29', 21),
 ('2010-06-28', 21),
 ('2012-01-19', 21),
 ('2011-09-01', 21),
 ('2010-10-28', 21),
 ('2010-07-08', 21),
 ('2013-05-03', 21),
 ('2012-02-23', 21),
 ('2013-04-30

In [3]:
example

{'Headline': 'Gold Rises to Record $1,498.90 an Ounce as Weakening Dollar Stokes Demand',
 'Journalists': ['Maria Kolesnikova', 'Kyoungwha Kim'],
 'Date': Timestamp('2011-04-19 12:22:50'),
 'Link': 'http://www.bloomberg.com/news/2011-04-19/gold-may-top-1-500-extending-rally-to-record-as-u-s-credit-outlook-cut.html',
 'Article': 'Gold extended gains to a record in New York as a drop in the dollar buoyed demand for the metal as an alternative investment. Futures surged yesterday after Standard & Poor’s revised its U.S. credit outlook to negative. Gold has jumped 5.3 percent this year as the dollar dropped 4.8 percent against a basket of six other currencies including the euro and British pound. Gold prices may keep rising for “some years into the future,” Blackrock Inc. (BLK) fund manager Evy Hambro said in an interview with Mark Barton on Bloomberg TV’s “On the Move.” Gold for June delivery rose $1.60, or 0.1 percent, to $1,494.60 an ounce by 8:20 a.m. in New York after earlier today cl

In [ ]:
counter

(10456, 1)

In [ ]:
_df = ds["train"].to_pandas()
for idx, row in _df.iterrows():
    print(row.get("Headline", ""))

In [2]:
from knowledge_qa_cibc import BloombergFinancialNewsDataset

BloombergFinancialNewsDataset()

In [3]:
from knowledge_qa_cibc import BloombergNewsExample

BloombergNewsExample()

ValidationError: 3 validation errors for BloombergNewsExample
example_id
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
title
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
content
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing